In [ ]:
import BAC0
import json
import paho.mqtt.client as mqtt
import time

# BACnet settings
bacnet_ip = '192.168.1.9/24'  # BACnet/IP address of your network interface
bacnet = BAC0.connect(ip=bacnet_ip)

# MQTT settings
mqtt_broker = '127.0.0.1'
mqtt_port = 1883
mqtt_topic = 'bacnet_data'

# BACnet device and properties to read
device_id = 1867817
property_id = 'presentValue'  # Example property

# MQTT client setup
mqtt_client = mqtt.Client()

def on_connect(client, userdata, flags, rc):
    print("Connected to MQTT Broker with result code " + str(rc))

def on_publish(client, userdata, result):
    print("Data published to MQTT topic")

mqtt_client.on_connect = on_connect
mqtt_client.on_publish = on_publish
mqtt_client.connect(mqtt_broker, mqtt_port, 60)

def read_bacnet_property(device_id, property_id):
    try:
        result = bacnet.read(f'{device_id} {property_id}')
        return result
    except Exception as e:
        print(f"Error reading BACnet property: {e}")
        return None

def publish_to_mqtt(data):
    json_data = json.dumps(data)
    mqtt_client.publish(mqtt_topic, json_data)

def main():
    while True:
        property_value = read_bacnet_property(device_id, property_id)
        if property_value is not None:
            data = {
                'device_id': device_id,
                'property_id': property_id,
                'value': property_value
            }
            publish_to_mqtt(data)
        time.sleep(10)  # Read and publish every 10 seconds

if __name__ == '__main__':
    try:
        main()
    except KeyboardInterrupt:
        print("Exiting...")
    finally:
        bacnet.disconnect()
        mqtt_client.disconnect()


In [ ]:
import asyncio
import BAC0
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# BACnet device configuration
BACNET_DEVICES = [
    {'deviceIp': '192.168.1.12', 'type': 'analogInput', 'objectId': '1'}
]

# Constants
MAX_RETRIES = 5

async def read_bacnet_value(bacnet, device):
    """
    Reads a BACnet object value with retry logic.
    """
    device_address = f"{device['deviceIp']} {device['type']} {device['objectId']} presentValue"

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            value = await bacnet.read(device_address)
            logger.info(f"Value read successfully from {device['deviceIp']}: {value}")
            return value  # Return the value after successful read
        except Exception as e:
            logger.error(f"Attempt {attempt}/{MAX_RETRIES} failed: {e}")
            if attempt < MAX_RETRIES:
                await asyncio.sleep(2)  # Wait before retrying
    logger.error(f"Failed to read value from {device['deviceIp']} after {MAX_RETRIES} attempts")
    return None

async def main():
    logger.info(f"BAC0 Version: {BAC0.infos.__version__}")

    # Start the BACnet connection
    async with BAC0.start(ip='192.168.1.12/24') as bacnet:
        tasks = [read_bacnet_value(bacnet, dev) for dev in BACNET_DEVICES]
        results = await asyncio.gather(*tasks)  # Read all devices concurrently
        logger.info("All readings completed")
        return results

# Adjust for environments with running loops
if __name__ == "__main__":
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    finally:
        loop.run_until_complete(main())
